# 获取底盘反馈信息

产品开机后，下位机默认会持续向上位机反馈各类信息，你可以通过这些反馈信息来获取目前产品的工作状态。

通常来说，你需要连续获取下位机反馈的信息，但在本例程中，我们只获取一个由下位机反馈的 JSON 信息（注释掉或删除 break 这行即可连续获取反馈信息）。

选中以下的代码块，使用 Ctrl + Enter 运行该代码块，当它获取第一条完整的且T值为 1001 的 JSON 信息后会跳出循环并输出反馈信息，反馈信息包含当前车轮转速、IMU、云台角度（如果有安装的话）、机械臂角度（如果有安装的话）、电源电压等信息。

In [ ]:
from base_ctrl import BaseController
import json
import time

base = BaseController('/dev/ttyTHS1', 115200)

# 使用无限循环来不断监听串口数据
while True:
    try:
        # 从串口读取一行数据，解码成 'utf-8' 格式的字符串，并尝试将其转换为 JSON 对象
        data_recv_buffer = json.loads(base.rl.readline().decode('utf-8'))
        # 检查解析出的数据中是否包含 'T' 键
        if 'T' in data_recv_buffer:
            # 如果 'T' 的值为 1001，则打印接收到的数据，并跳出循环
            if data_recv_buffer['T'] == 1001:
                print(data_recv_buffer)
                break
    # 如果在读取或处理数据时发生异常，则忽略该异常并继续监听下一行数据
    except:
        pass

## 非堵塞方式获取串口发来的 JSON 信息

以下的代码仅用于理解底层的串口读取 JSON 信息的原理，不可以执行下面的代码块。

In [ ]:
class ReadLine:
    # 构造函数，初始化 ReadLine 类的实例
    # s: 传入的串口对象，用于与串口进行通信。
    def __init__(self, s):
        self.buf = bytearray()  # 初始化一个字节数组，用于存储从串口读取但尚未处理的数据
        self.s = s  # 保存传入的串口对象，后续用于读取串口数据
        self.timeout = 0.1                  # 设置超时时间（秒），超过该时间未接收到完整数据帧则返回 None
        self.frame_start = b'{'              # 定义数据帧起始标志
        self.frame_end = b"}\r\n"            # 定义数据帧结束标志
        self.max_frame_length = 512          # 定义最大帧长度，防止内存溢出或异常数据阻塞

    def readline(self):
        start_time = time.time()             # 记录开始读取的时间，用于超时判断
        while True:
            # 从串口读取尽可能多的数据，但不超过 max_frame_length
            # 同时保证每次读取至少 1 字节
            i = max(1, min(self.max_frame_length, self.s.in_waiting))
            data = self.s.read(i)            # 读取 i 个字节
            if data:
                self.buf.extend(data)        # 将读取到的数据追加到缓冲区

            # 在缓冲区中查找结束标志
            end = self.buf.rfind(self.frame_end)

            if end >= 0:  # 找到了结束标志
                # 在结束标志之前寻找起始标志
                start = self.buf.rfind(self.frame_start, 0, end)
                if start >= 0 and start < end:
                    # 提取完整的数据帧（包括起止标志）
                    r = self.buf[start:end + len(self.frame_end)]
                    # 从缓冲区中移除已经处理过的数据
                    self.buf = self.buf[end + len(self.frame_end):]
                    return r
                elif start == -1:
                    # 没找到起始标志，继续接收数据
                    continue

            # 超时处理
            if time.time() - start_time > self.timeout:
                break

该方法用于从串口中按帧读取数据，每帧数据以 `{` 开头，以 `}\r\n` 结尾。

* 方法会先检查缓冲区中是否已包含完整的帧数据，如果有则直接提取并返回。
* 如果缓冲区中没有完整帧，则通过 `in_waiting` 获取串口缓冲区可读字节数（最大不超过 512 字节），并将读取到的数据追加到缓冲区中。
* 每次读取后，会查找帧结束标志 `}\r\n`，如果找到则继续向前查找帧起始标志 `{`，在起始和结束标志都存在且顺序正确时，提取这一段作为完整帧返回，并清理缓冲区中已处理的部分。
* 如果在设定的超时时间（0.1 秒）内仍未找到完整帧，则返回 `None`。

### 函数特性

* **帧解析能力强**：能够准确识别以 `{` 开头、以 `}\r\n` 结尾的完整数据帧，确保解析的数据完整且格式正确。
* **非阻塞读取**：采用短超时和按需读取方式，即使没有可读数据也不会长时间阻塞程序运行，适合实时系统。
* **高效缓冲管理**：通过最大 512 字节的分段读取和动态缓冲区拼接，有效减少内存占用并避免数据丢失。
* **容错性好**：能够处理缓冲区内残留的不完整数据，并在新数据到达后自动拼接与解析，保证连续性。
* **实时性强**：每次读取后立即检测并提取完整帧，适用于高频率的串口通信场景。
* **适配结构化数据**：对 JSON 等结构化帧格式的读取、提取和返回进行了优化，减少后续数据处理的复杂度。